# Solution to PERONA LAB SURF task - Anna Rutkiewicz


# Dependencies

### Install required dependencies

In [ ]:
# !pip install --upgrade torchvision
# !pip install lightning

# Caltech101 Dataset

### Initialize Caltech101DataModule

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import Caltech101

from transformers import CLIPModel, CLIPProcessor
import lightning as L

class Caltech101DataModule(L.LightningDataModule):
    def __init__(self, data_dir="./data", batch_size=32, num_workers=4, collate_fn=None):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

        self._collate_fn = collate_fn
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.num_classes = None
        self.class_names = None

    def prepare_data(self):
        Caltech101(root=self.data_dir, download=True)

    def setup(self, stage=None):
        full = Caltech101(root=self.data_dir, download=False)
        self.num_classes = len(full.categories)
        self.class_names = list(full.categories)
        n_total = len(full)
        n_train = int(0.7 * n_total)
        n_val = int(0.2 * n_total)
        n_test = n_total - n_train - n_val
        self.train_dataset, self.val_dataset, self.test_dataset = random_split(
            full, [n_train, n_val, n_test],
            generator=torch.Generator().manual_seed(12),
        )

    def train_dataloader(self):
        if self._collate_fn is None:
          raise ValueError("collate_fn not set")
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            collate_fn=self._collate_fn,
            pin_memory=True,
        )

    def val_dataloader(self):
        if self._collate_fn is None:
          raise ValueError("collate_fn not set")
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            collate_fn=self._collate_fn,
            pin_memory=True,
        )

    def test_dataloader(self):
        if self._collate_fn is None:
          raise ValueError("collate_fn not set")
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            collate_fn=self._collate_fn,
            pin_memory=True,
        )

# CLIP

### Define CLIPClassifier
- CLIP weights frozen
- lightweight neural network trained on CLIP embeddings

In [ ]:
class CLIPClassifier(L.LightningModule):
    def __init__(
        self,
        num_classes: int,
        lr: float = 1e-3,
        clip_model_name: str = "openai/clip-vit-base-patch32",
        freeze_clip: bool = True,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.clip = CLIPModel.from_pretrained(clip_model_name)
        self.processor = CLIPProcessor.from_pretrained(clip_model_name)
        if freeze_clip:
          self.clip.eval()
          for p in self.clip.parameters():
              p.requires_grad = False
        embed_dim = self.clip.config.projection_dim
        # classifier on top of CLIP embeddings
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes),
        )
        self.criterion = nn.CrossEntropyLoss()

    def _unpack_batch(self, batch):
        if isinstance(batch, dict):
            pixel_values = batch.get("pixel_values", None)
            labels = batch.get("labels", None)
        else:
            pixel_values, labels = batch
        if pixel_values is None or labels is None:
            raise ValueError("Batch must contain 'pixel_values' and 'labels'.")
        return pixel_values, labels

    def forward(self, pixel_values):
        # pixel_values: [B, 3, 224, 224]
        with torch.no_grad():
            feats = self.clip.get_image_features(pixel_values=pixel_values)
        feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
        logits = self.classifier(feats)
        return logits

    def _shared_step(self, batch, stage: str):
        pixel_values, labels = self._unpack_batch(batch=batch)
        pixel_values = pixel_values.to(self.device, non_blocking=True)
        labels = labels.to(self.device, dtype=torch.long, non_blocking=True)
        logits = self(pixel_values)
        loss = self.criterion(logits, labels)
        preds = logits.argmax(dim=-1)
        acc = (preds == labels).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True)
        self.log(f"{stage}_acc", acc, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.classifier.parameters(), lr=self.hparams.lr)
        return optimizer

    def collate_fn(self, batch):
        images, labels = zip(*batch)
        inputs = self.processor(images=list(images), return_tensors="pt")
        pixel_values = inputs["pixel_values"]
        labels = torch.tensor(labels, dtype=torch.long)
        return pixel_values, labels


### Classification - CLIP

In [ ]:
import gc

CALTECH101_NUM_CLASSES = 101

def train_classifier_clip():
    model = CLIPClassifier(num_classes=CALTECH101_NUM_CLASSES, lr=1e-3)
    dm = Caltech101DataModule(
        data_dir="./data",
        batch_size=64,
        num_workers=4,
        collate_fn=model.collate_fn,
    )
    dm.prepare_data()
    dm.setup("fit")
    trainer = L.Trainer(
        max_epochs=10,
        accelerator="auto",
        devices=1,
        precision="16-mixed",  # faster on GPU
    )
    trainer.fit(model, datamodule=dm)
    trainer.test(model, datamodule=dm)
    cleanup_clip(model, datamodule=dm)

def cleanup_clip(model, datamodule):
  del model
  del dm.processor
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

if __name__ == "__main__":
  train_classifier_clip()

# LLAVA

### Define LLAVA ZeroShot Classifier

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration

def llava_collate(batch):
    images, labels = zip(*batch)
    return {"images": list(images), "labels": torch.tensor(labels, dtype=torch.long)}

class LlavaZeroShotClassifier(L.LightningModule):
    def __init__(
        self,
        class_names,
        llava_model_name="llava-hf/llava-1.5-7b-hf",
        max_new_tokens=32,
        freeze_llava=True,
    ):
        super().__init__()
        self.class_names = list(class_names)
        self.max_new_tokens = max_new_tokens
        self.processor = AutoProcessor.from_pretrained(llava_model_name)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            llava_model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        if hasattr(self.processor, "tokenizer"):
            self.processor.tokenizer.padding_side = "left"
        if freeze_llava:
          self.model.eval()
          for p in self.model.parameters():
              p.requires_grad = False
        self.class_list_str = ", ".join(self.class_names)

    @torch.no_grad()
    def _build_messages(self):
        user_text = (
            "You are an image classifier. "
            "Choose the single best category for this image from the following list:\n"
            f"{self.class_list_str}\n"
            "Answer with exactly one category name from the list."
        )
        return [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": user_text},
                ],
            }
        ]

    @torch.no_grad()
    def _extract_class_idx(self, output_text: str):
        t = output_text.lower()
        for i, name in enumerate(self.class_names):
            if name.lower() in t:
                return i
        return 0

    @torch.no_grad()
    def _predict_batch(self, images):
        base_messages = self._build_messages()
        texts = [
            self.processor.apply_chat_template(
                base_messages, add_generation_prompt=True
            )
            for _ in range(len(images))
        ]
        inputs = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
        ).to(self.device)
        out_ids = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
        )
        outputs = self.processor.batch_decode(
            out_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        pred_idxs = [self._extract_class_idx(o) for o in outputs]
        return torch.tensor(pred_idxs, device=self.device, dtype=torch.long)

    def validation_step(self, batch, _idx):
        preds = self._predict_batch(batch["images"])
        labels = batch["labels"].to(self.device, dtype=torch.long)
        acc = (preds == labels).float().mean()
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, _idx):
        preds = self._predict_batch(batch["images"])
        labels = batch["labels"].to(self.device, dtype=torch.long)
        acc = (preds == labels).float().mean()
        self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)

    # No optimizer needed
    def configure_optimizers(self):
        return None


### Classification - LLAVA

In [ ]:
import torch

def run_llava_classification():
  dm = Caltech101DataModule(
      data_dir="./data",
      batch_size=4,
      num_workers=4,
      collate_fn=llava_collate,
  )
  dm.prepare_data()
  dm.setup("fit")
  model = LlavaZeroShotClassifier(class_names=dm.class_names)
  trainer = L.Trainer(
      max_epochs=None,
      accelerator="auto",
      devices=1,
      precision="16-mixed" if torch.cuda.is_available() else "32-true",
      log_every_n_steps=1,
  )
  trainer.test(model, datamodule=dm)

if __name__ == "__main__":
  run_llava_classification()